In [1]:
import re
import numpy as np
import pandas as pd

In [2]:
def clean_text(text: str) -> str:
    """Normalize whitespace."""
    text = re.sub(r"\s+", " ", text)
    return text.strip()

In [3]:
def to_list(value):
    """
    Convert numpy arrays / lists / None into
    a clean list of strings.
    """

    if value is None:
        return []

    if isinstance(value, float) and pd.isna(value):
        return []

    if isinstance(value, np.ndarray):
        value = value.tolist()

    if isinstance(value, list):
        cleaned = []

        for item in value:
            if item is None:
                continue

            text = clean_text(str(item))

            if text:
                cleaned.append(text)

        return cleaned

    text = clean_text(str(value))

    return [text] if text else []

In [4]:
def format_title(row):
    title = clean_text(str(row.get("title", "")))
    return f"Title: {title}"

def format_brand(row) -> str:

    brand = row.get("store")

    if pd.isna(brand) is True:
        return ""

    return f"Brand: {clean_text(str(brand))}"

def format_categories(row: pd.Series) -> str:

    categories = to_list(row.get("categories"))

    if not categories:
        return ""

    return "Categories: " + " > ".join(categories)

def format_features(row: pd.Series) -> str:

    features = to_list(row.get("features"))

    if not features:
        return ""

    bullets = "\n".join(f"- {feature}" for feature in features)

    return f"Features:\n{bullets}"

def format_description(row: pd.Series) -> str:

    description = to_list(row.get("description"))

    if not description:
        return ""

    text = " ".join(description)

    if len(text) > 200:  # Assuming a default max_description_chars value
        text = text[:200].rstrip()

    return f"Description:\n{text}"

def transform_product_row_to_textual_discription(row):
    
    sections = [format_title(row)]

    sections.append(format_brand(row))

    sections.append(format_categories(row))

    sections.append(format_features(row))

    sections.append(format_description(row))

    return "\n\n".join(
        section
        for section in sections
        if section.strip()
    )

In [5]:
def get_image_url(images):
    """Return the first available image URL."""

    if not isinstance(images, dict):
        return None

    for key in ["large", "hi_res", "thumb"]:
        urls = images.get(key)

        if urls is None:
            continue

        # Convert numpy array to list
        if isinstance(urls, np.ndarray):
            urls = urls.tolist()

        for url in urls:
            if url is not None and url != "":
                return url

    return None

In [6]:
def build_metadata(row: pd.Series) -> dict:
    
    return {
        "parent_asin": row.get("parent_asin"),
        "title": row.get("title"),
        "brand": row.get("store"),
        "categories": to_list(row.get("categories")),
        "price": row.get("price"),
        "rating": row.get("average_rating"),
        "rating_count": row.get("rating_number"),
        'image_url': get_image_url(row["images"])
    }

In [7]:
from datasets import load_dataset

# from src.preprocessing import (
#     build_metadata,ß≈ß
#     transform_product_row_to_textual_discription,
#     build_corpus
# )

/Users/tusharrohilla/Desktop/natural_language_search_engine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
dataset = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_meta_Electronics",
    split="full"
)

Using the latest cached version of the dataset since McAuley-Lab/Amazon-Reviews-2023 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'raw_meta_Electronics' at /Users/tusharrohilla/.cache/huggingface/datasets/McAuley-Lab___amazon-reviews-2023/raw_meta_Electronics/0.0.0/16b76e0823d73bb8cff1e9c5e3e37dbc46ae3daee380417ae141f5e67d3ea8e8 (last modified on Tue Aug  4 08:05:44 2026).


In [9]:
df = dataset.to_pandas()

In [10]:
df.columns

Index(['main_category', 'title', 'average_rating', 'rating_number', 'features',
       'description', 'price', 'images', 'videos', 'store', 'categories',
       'details', 'parent_asin', 'bought_together', 'subtitle', 'author'],
      dtype='str')

In [11]:
row = df.iloc[0]

In [12]:
row

main_category                                        All Electronics
title                         FS-1051 FATSHARK TELEPORTER V3 HEADSET
average_rating                                                   3.5
rating_number                                                      6
features                                                          []
description        [Teleporter V3 The “Teleporter V3” kit sets a ...
price                                                           None
images             {'hi_res': [None], 'large': ['https://m.media-...
videos                       {'title': [], 'url': [], 'user_id': []}
store                                                      Fat Shark
categories          [Electronics, Television & Video, Video Glasses]
details            {"Date First Available": "August 2, 2014", "Ma...
parent_asin                                               B00MCW7G9M
bought_together                                                  NaN
subtitle                          

In [13]:
row_text = transform_product_row_to_textual_discription(row)

In [14]:
print(row_text)

Title: FS-1051 FATSHARK TELEPORTER V3 HEADSET

Brand: Fat Shark

Categories: Electronics > Television & Video > Video Glasses

Description:
Teleporter V3 The “Teleporter V3” kit sets a new level of value in the FPV world with Fat Shark renowned performance and quality. The fun of FPV is experienced firsthand through the large screen FPV h


In [15]:
type(row_text)

str

In [16]:
build_metadata(row)

{'parent_asin': 'B00MCW7G9M',
 'title': 'FS-1051 FATSHARK TELEPORTER V3 HEADSET',
 'brand': 'Fat Shark',
 'categories': ['Electronics', 'Television & Video', 'Video Glasses'],
 'price': 'None',
 'rating': np.float64(3.5),
 'rating_count': np.int64(6),
 'image_url': 'https://m.media-amazon.com/images/I/41qrX56lsYL._AC_.jpg'}